# Medical Framework — Unified Pipeline
Single `imblearn.Pipeline` searched by `GridSearchCV` over a pre-sampled
Latin Hypercube grid (200 candidates with per-family quotas). Each step is
one of the custom transformers from the `.py` modules; the candidate grid
swaps each step out per draw.

In [1]:
import os
import shutil
import warnings
import joblib
import numpy as np
import pandas as pd

warnings.filterwarnings('ignore')

from imblearn.pipeline import Pipeline
from imblearn import FunctionSampler
from sklearn.model_selection import GridSearchCV, StratifiedKFold, train_test_split
from sklearn.metrics import (
    roc_auc_score, average_precision_score, classification_report,
    confusion_matrix, precision_recall_curve, roc_curve, f1_score,
)
from sklearn.calibration import CalibratedClassifierCV
from sklearn.ensemble import StackingClassifier
from sklearn.linear_model import LogisticRegression

from loader import load_data

from clean_data import MeanImputer, KNNImputerWrapper, IterativeModelImputer, DropRowsImputer, ClassMeanImputer, InterpolateImputer
from tame_outlier import OutlierTamer
from normalization import RobustScalerNorm, ZScoreNormalizationNorm
from feature_selection import SelectKBestFilter, TreeBasedSelection
from balance import SMOTESampler, BorderlineSMOTESampler
from model_training import (
    LogisticRegressionEstimator, RandomForestEstimator,
    XGBoostEstimator, LightGBMEstimator, CatBoostEstimator,
)

# imblearn-native no-op sampler — replaces the custom IdentitySampler whose
# `_sampling_type = "bypass"` was the most likely culprit for the NaN-everywhere
# CV scores. FunctionSampler with a pass-through func is officially supported.
def _identity(X, y):
    return X, y

def make_identity_sampler():
    return FunctionSampler(func=_identity, validate=False)

# Clear any stale joblib pipeline cache from previous failed runs. A poisoned
# cache combined with n_jobs>1 was the second suspect for the NaN scores.
shutil.rmtree('./cache', ignore_errors=True)


## Load & split

In [2]:
path = './CardioKaggle/cardio_train'
typeData = 'csv'
y_column = 'cardio'

X, Y, all_mappings, y_mappings = load_data(path=f'{path}.{typeData}', y_column=y_column)
X = X.astype('float32')

print(f'X shape : {X.shape}')
print(f'Classes : {pd.Series(Y).value_counts().to_dict()}')

x_train, x_val, y_train, y_val = train_test_split(
    X, Y, test_size=0.2, random_state=42, stratify=Y,
)

# Imbalance ratio used to seed scale_pos_weight searches downstream.
neg, pos = (np.array(y_train) == 0).sum(), (np.array(y_train) == 1).sum()
spw_base = float(neg) / float(max(pos, 1))
print(f'neg/pos in train: {neg}/{pos}  ->  scale_pos_weight base ~ {spw_base:.2f}')

X shape : (70000, 12)
Classes : {0: 35021, 1: 34979}
neg/pos in train: 28017/27983  ->  scale_pos_weight base ~ 1.00


## Build the pipeline
Six stages: imputation → outlier flags → normalization → feature selection → balancing → classifier. The starting values are placeholders — `RandomizedSearchCV` swaps each step out below.

In [3]:
from sklearn.base import is_classifier

pipeline = Pipeline(steps=[
    ('imputer',    MeanImputer()),
    ('tamer',      OutlierTamer()),
    ('normalizer', ZScoreNormalizationNorm()),
    ('selector',   SelectKBestFilter(k=20)),
    ('balancer',   make_identity_sampler()),
    ('classifier', LogisticRegressionEstimator()),
    ]
    ,memory='./cache')
# NOTE: memory='./cache' removed intentionally. A stale joblib cache with
# n_jobs>1 was a prime suspect for the NaN scores. Re-enable later if needed.

# Sanity guard — the original NaN cascade was caused by sklearn treating the
# wrapped classifiers as regressors, which made every roc_auc / pr_auc scorer
# raise inside CV. Catch the misconfiguration here, before a 1000-fit sweep.
assert is_classifier(pipeline), (
    'Pipeline is not recognized as a classifier. Check that the final-step '
    'estimator inherits as (ClassifierMixin, BaseEstimator) — mixin first.'
)
pipeline

,steps,"[('imputer', ...), ('tamer', ...), ...]"
,transform_input,None
,memory,'./cache'
,verbose,False
,detector,'iforest'
,remediation,'winsorize'
,contamination,0.05
,z_threshold,3.0
,mad_threshold,3.5
,lower_quantile,0.01
,upper_quantile,0.99


## Search space
Each sub-dict pins one classifier and lists compatible step choices + hyper-parameters. `RandomizedSearchCV` samples 200 combinations from the cross-product of all sub-dicts.

In [4]:
from build_param_grid import build_param_grid, diversity_report

TOTAL_N = 200   # candidates pre-sampled via Latin Hypercube + quotas
param_grid, family_index = build_param_grid(spw_base, total_n=TOTAL_N, seed=42)
print(f'Pre-sampled {len(param_grid)} candidates across {len(set(family_index))} model families.')


Pre-sampled 200 candidates across 6 model families.


In [5]:
# Diversity audit — entropy across categorical axes and spread across
# continuous axes, computed BEFORE any fitting. Flags axes with H<0.80.
_ = diversity_report(param_grid, family_index)


=== Family distribution ===
  catboost     20   (10.0%)
  lgbm         60   (30.0%)
  lr           20   (10.0%)
  rf           20   (10.0%)
  stacker      20   (10.0%)
  xgb          60   (30.0%)

=== Categorical entropy (Hₙ ∈ [0,1]; 1.0 = uniform across categories) ===
  _family                              H=0.917   k= 6   n=200
  imputer                              H=1.000   k= 5   n=200
  normalizer                           H=1.000   k= 2   n=200
  tamer__detector                      H=1.000   k= 3   n=200
  tamer__remediation                   H=1.000   k= 6   n=200
  selector                             H=1.000   k= 5   n=200
  balancer                             H=1.000   k= 4   n=200
  classifier__max_features             H=0.998   k= 3   n= 20
  classifier__penalty                  H=1.000   k= 2   n= 20

=== Continuous spread (per-axis; only over candidates that sample it) ===
  axis                                                  min         max        std     n
  class

## Fit the search

In [6]:
import traceback
from sklearn.model_selection import cross_validate

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# ---------------------------------------------------------------------------
# Stage 1: smoke test on the default pipeline (no grid overrides).
# Confirms imputer → tamer → normalizer → selector → balancer → classifier
# wiring before spending real compute on the full sweep.
# ---------------------------------------------------------------------------
# print('--- smoke test (default pipeline, error_score=raise) ---')
# try:
#     smoke = cross_validate(
#         pipeline, x_train, y_train,
#         scoring={'pr_auc': 'average_precision', 'roc_auc': 'roc_auc'},
#         cv=cv, n_jobs=1, error_score='raise',
#         return_train_score=False,
#     )
#     print(f"  pr_auc  mean: {np.mean(smoke['test_pr_auc']):.4f}")
#     print(f"  roc_auc mean: {np.mean(smoke['test_roc_auc']):.4f}")
#     smoke_ok = True
# except Exception:
#     print('SMOKE TEST FAILED — do not run the full search yet.')
#     traceback.print_exc()
#     smoke_ok = False

# ---------------------------------------------------------------------------
# Stage 2: GridSearchCV over the pre-sampled candidates.
# Each dict in `param_grid` already pins one full configuration (every value
# wrapped in a length-1 list), so GridSearchCV evaluates each one exactly once.
# error_score=np.nan keeps pathological combos (e.g. SMOTE k_neighbors >
# minority size in a fold) from killing the whole sweep; Stage 3 surfaces them.
# ---------------------------------------------------------------------------
smoke_ok=True
search = None
if smoke_ok:
    search = GridSearchCV(
        estimator=pipeline,
        param_grid=param_grid,
        scoring={'pr_auc': 'average_precision', 'roc_auc': 'roc_auc'},
        refit='pr_auc',
        cv=cv,
        n_jobs=-1,
        verbose=2,
        return_train_score=False,
        error_score=np.nan,
    )
    try:
        search.fit(x_train, y_train)
    except Exception:
        print('GridSearchCV.fit raised — full traceback below.')
        traceback.print_exc()
        search = None

# ---------------------------------------------------------------------------
# Stage 3: surface silent failures (NaN scores) inside cv_results_, and report
# PR-AUC dispersion — the metric that tells you whether the wider search
# actually produced wider results.
# ---------------------------------------------------------------------------
if search is not None and hasattr(search, 'cv_results_'):
    res = pd.DataFrame(search.cv_results_)
    n_total  = len(res)
    n_failed = int(res['mean_test_pr_auc'].isna().sum())
    print(f'\n--- post-search audit ---')
    print(f'configs total : {n_total}')
    print(f'configs OK    : {n_total - n_failed}')
    print(f'configs NaN   : {n_failed}')
    if n_failed and 'fit_error' in res.columns:
        msgs = (
            res.loc[res['mean_test_pr_auc'].isna(), 'fit_error']
               .dropna().astype(str).str.slice(0, 240).value_counts().head(5)
        )
        if len(msgs):
            print('\nTop failure messages (truncated):')
            for msg, n in msgs.items():
                print(f'  [{n}x] {msg}')

    pr = pd.to_numeric(res['mean_test_pr_auc'], errors='coerce').dropna()
    if len(pr):
        print(f'\n--- PR-AUC dispersion across candidates ---')
        print(f'  min    : {pr.min():.4f}')
        print(f'  p25    : {pr.quantile(0.25):.4f}')
        print(f'  median : {pr.median():.4f}')
        print(f'  p75    : {pr.quantile(0.75):.4f}')
        print(f'  max    : {pr.max():.4f}')
        print(f'  std    : {pr.std():.4f}    range: {pr.max() - pr.min():.4f}')


Fitting 5 folds for each of 200 candidates, totalling 1000 fits


/home/sinam/Python/miniconda3/envs/qenv/lib/python3.12/site-packages/xgboost/core.py:377: FutureWarning: Your system has an old version of glibc (< 2.28). We will stop supporting Linux distros with glibc older than 2.28 after **May 31, 2025**. Please upgrade to a recent Linux distro (with glibc >= 2.28) to use future versions of XGBoost.
Note: You have installed the 'manylinux2014' variant of XGBoost. Certain features such as GPU algorithms or federated learning are not available. To use these features, please upgrade to a recent Linux distro with glibc 2.28+, and install the 'manylinux_2_28' variant.
  warnings.warn(
/home/sinam/Python/miniconda3/envs/qenv/lib/python3.12/site-packages/xgboost/core.py:377: FutureWarning: Your system has an old version of glibc (< 2.28). We will stop supporting Linux distros with glibc older than 2.28 after **May 31, 2025**. Please upgrade to a recent Linux distro (with glibc >= 2.28) to use future versions of XGBoost.
Note: You have installed the 'many

/home/sinam/Python/miniconda3/envs/qenv/lib/python3.12/site-packages/xgboost/core.py:377: FutureWarning: Your system has an old version of glibc (< 2.28). We will stop supporting Linux distros with glibc older than 2.28 after **May 31, 2025**. Please upgrade to a recent Linux distro (with glibc >= 2.28) to use future versions of XGBoost.
Note: You have installed the 'manylinux2014' variant of XGBoost. Certain features such as GPU algorithms or federated learning are not available. To use these features, please upgrade to a recent Linux distro with glibc 2.28+, and install the 'manylinux_2_28' variant.
  warnings.warn(
/home/sinam/Python/miniconda3/envs/qenv/lib/python3.12/site-packages/xgboost/core.py:377: FutureWarning: Your system has an old version of glibc (< 2.28). We will stop supporting Linux distros with glibc older than 2.28 after **May 31, 2025**. Please upgrade to a recent Linux distro (with glibc >= 2.28) to use future versions of XGBoost.
Note: You have installed the 'many

/home/sinam/Python/miniconda3/envs/qenv/lib/python3.12/site-packages/xgboost/core.py:377: FutureWarning: Your system has an old version of glibc (< 2.28). We will stop supporting Linux distros with glibc older than 2.28 after **May 31, 2025**. Please upgrade to a recent Linux distro (with glibc >= 2.28) to use future versions of XGBoost.
Note: You have installed the 'manylinux2014' variant of XGBoost. Certain features such as GPU algorithms or federated learning are not available. To use these features, please upgrade to a recent Linux distro with glibc 2.28+, and install the 'manylinux_2_28' variant.
  warnings.warn(


[CV] END balancer=BorderlineSMOTESampler(), classifier=XGBoostEstimator(), classifier__colsample_bytree=0.8419374188963815, classifier__learning_rate=0.05463642015934537, classifier__max_depth=9, classifier__min_child_weight=27, classifier__n_estimators=333, classifier__reg_alpha=0.024837040528698836, classifier__reg_lambda=0.28249762844686366, classifier__scale_pos_weight=1.4207305907300938, classifier__subsample=0.9521269820341016, imputer=IterativeModelImputer(), normalizer=RobustScalerNorm(), selector=SelectKBestFilter(), tamer__contamination=0.03854465238600047, tamer__detector=iforest, tamer__mad_threshold=2.670229295955927, tamer__remediation=mad_replace; total time=  16.7s
[CV] END balancer=SMOTESampler(k_neighbors=5), classifier=XGBoostEstimator(), classifier__colsample_bytree=0.8891337491828115, classifier__learning_rate=0.11598157060224928, classifier__max_depth=9, classifier__min_child_weight=2, classifier__n_estimators=174, classifier__reg_alpha=0.021348412848947847, class

...

/home/sinam/Python/miniconda3/envs/qenv/lib/python3.12/site-packages/xgboost/core.py:377: FutureWarning: Your system has an old version of glibc (< 2.28). We will stop supporting Linux distros with glibc older than 2.28 after **May 31, 2025**. Please upgrade to a recent Linux distro (with glibc >= 2.28) to use future versions of XGBoost.
Note: You have installed the 'manylinux2014' variant of XGBoost. Certain features such as GPU algorithms or federated learning are not available. To use these features, please upgrade to a recent Linux distro with glibc 2.28+, and install the 'manylinux_2_28' variant.
  warnings.warn(


/home/sinam/Python/miniconda3/envs/qenv/lib/python3.12/site-packages/xgboost/core.py:377: FutureWarning: Your system has an old version of glibc (< 2.28). We will stop supporting Linux distros with glibc older than 2.28 after **May 31, 2025**. Please upgrade to a recent Linux distro (with glibc >= 2.28) to use future versions of XGBoost.
Note: You have installed the 'manylinux2014' variant of XGBoost. Certain features such as GPU algorithms or federated learning are not available. To use these features, please upgrade to a recent Linux distro with glibc 2.28+, and install the 'manylinux_2_28' variant.
  warnings.warn(



[CV] END balancer=BorderlineSMOTESampler(), classifier=XGBoostEstimator(), classifier__colsample_bytree=0.7581264739016785, classifier__learning_rate=0.28946162301935163, classifier__max_depth=6, classifier__min_child_weight=2, classifier__n_estimators=103, classifier__reg_alpha=0.002921219848692909, classifier__reg_lambda=0.48949564347055824, classifier__scale_pos_weight=1.1328372604146246, classifier__subsample=0.9484033082528108, imputer=ClassMeanImputer(), normalizer=ZScoreNormalizationNorm(), selector=SelectKBestFilter(k=30), tamer__contamination=0.1377819581453969, tamer__detector=cluster, tamer__mad_threshold=3.1489529652998476, tamer__remediation=mad_replace; total time=  44.6s
[CV] END balancer=BorderlineSMOTESampler(), classifier=XGBoostEstimator(), classifier__colsample_bytree=0.9426120203152344, classifier__learning_rate=0.10632252403433884, classifier__max_depth=6, classifier__min_child_weight=2, classifier__n_estimators=257, classifier__reg_alpha=4.286392260105291, class

GridSearchCV.fit raised — full traceback below.


Traceback (most recent call last):
  File "/tmp/ipykernel_520667/3460263102.py", line 49, in <module>
    search.fit(x_train, y_train)
  File "/home/sinam/Python/miniconda3/envs/qenv/lib/python3.12/site-packages/sklearn/base.py", line 1365, in wrapper
    return fit_method(estimator, *args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/sinam/Python/miniconda3/envs/qenv/lib/python3.12/site-packages/sklearn/model_selection/_search.py", line 1051, in fit
    self._run_search(evaluate_candidates)
  File "/home/sinam/Python/miniconda3/envs/qenv/lib/python3.12/site-packages/sklearn/model_selection/_search.py", line 1605, in _run_search
    evaluate_candidates(ParameterGrid(self.param_grid))
  File "/home/sinam/Python/miniconda3/envs/qenv/lib/python3.12/site-packages/sklearn/model_selection/_search.py", line 997, in evaluate_candidates
    out = parallel(
          ^^^^^^^^^
  File "/home/sinam/Python/miniconda3/envs/qenv/lib/python3.12/site-packages/sklearn/utils/

## Inspect the winner

In [7]:
def _bail(msg):
    print(f'[skipped] {msg}')

# Guard: only run if the search actually fitted.
if search is None or not hasattr(search, 'best_estimator_'):
    _bail('No fitted search available. Fix errors reported in the previous cell and re-run.')
else:
    print(f'Best CV PR-AUC : {search.best_score_:.4f}')
    print('Best pipeline   :')
    for name, step in search.best_estimator_.named_steps.items():
        label = 'passthrough' if isinstance(step, str) else type(step).__name__
        print(f'  {name:11s} -> {label}')

    print('\nBest params:')
    for k, v in search.best_params_.items():
        print(f'  {k}: {v}')

    # Hold-out scores at the *default* 0.5 threshold (kept for continuity).
    y_proba = search.predict_proba(x_val)[:, 1]
    y_pred_default = (y_proba >= 0.5).astype(int)

    val_roc_auc = roc_auc_score(y_val, y_proba)
    val_pr_auc  = average_precision_score(y_val, y_proba)

    print(f'\nHold-out ROC-AUC : {val_roc_auc:.4f}')
    print(f'Hold-out PR-AUC  : {val_pr_auc:.4f}')
    print('\nConfusion matrix @ threshold=0.50 (uninformative on imbalanced data):')
    print(confusion_matrix(y_val, y_pred_default))
    print(classification_report(y_val, y_pred_default, zero_division=0))

    # ----------------------------------------------------------------------
    # Post-hoc threshold optimization.
    # A model that predicts at 0.5 on 11% prevalence is structurally biased
    # toward the majority class. We pick the threshold that maximizes F1 on
    # the holdout's precision-recall curve (swap to F2 if recall matters more
    # clinically).
    # ----------------------------------------------------------------------
    precisions, recalls, thresholds = precision_recall_curve(y_val, y_proba)
    f1s = 2 * precisions[:-1] * recalls[:-1] / np.clip(precisions[:-1] + recalls[:-1], 1e-12, None)
    best_idx = int(np.nanargmax(f1s))
    best_thr = float(thresholds[best_idx])

    beta = 2.0
    f2s = (1 + beta**2) * precisions[:-1] * recalls[:-1] / np.clip(beta**2 * precisions[:-1] + recalls[:-1], 1e-12, None)
    best_f2_idx = int(np.nanargmax(f2s))
    best_f2_thr = float(thresholds[best_f2_idx])

    fpr, tpr, roc_thr = roc_curve(y_val, y_proba)
    spec90_mask = (1 - fpr) >= 0.90
    if spec90_mask.any():
        idx90 = int(np.argmax(tpr * spec90_mask))
        recall_at_spec90 = float(tpr[idx90])
        thr_at_spec90    = float(roc_thr[idx90])
    else:
        recall_at_spec90, thr_at_spec90 = float('nan'), float('nan')

    print('\n--- Operating points ---')
    print(f'Best F1 threshold : {best_thr:.4f}   (F1={f1s[best_idx]:.3f}, P={precisions[best_idx]:.3f}, R={recalls[best_idx]:.3f})')
    print(f'Best F2 threshold : {best_f2_thr:.4f}  (F2={f2s[best_f2_idx]:.3f}, P={precisions[best_f2_idx]:.3f}, R={recalls[best_f2_idx]:.3f})')
    print(f'Recall @ Spec=0.90: {recall_at_spec90:.3f}  (threshold={thr_at_spec90:.4f})')

    y_pred_tuned = (y_proba >= best_thr).astype(int)
    print(f'\nConfusion matrix @ tuned F1 threshold={best_thr:.4f}:')
    print(confusion_matrix(y_val, y_pred_tuned))
    print(classification_report(y_val, y_pred_tuned, zero_division=0))

[skipped] No fitted search available. Fix errors reported in the previous cell and re-run.


## Leaderboard

In [8]:
if search is None or not hasattr(search, 'cv_results_'):
    print('[skipped] No cv_results_ available (fit failed).')
    cv_df = None
else:
    cv_df = (
        pd.DataFrame(search.cv_results_)
          .sort_values('mean_test_pr_auc', ascending=False)
          [['mean_test_pr_auc', 'std_test_pr_auc', 'mean_test_roc_auc', 'std_test_roc_auc', 'params']]
          .head(15)
          .reset_index(drop=True)
    )
    pd.DataFrame(search.cv_results_).to_csv("all_fits.csv",index=False)
cv_df

[skipped] No cv_results_ available (fit failed).


## Surrogate meta-table
Freeze the exploration sweep into a structured row-per-pipeline table — encoded preprocessing/selector/balancer/family choices and continuous hyperparameters as features; mean/std PR-AUC, mean/std ROC-AUC, fold-range, and an uncertainty-aware composite as multi-output targets. Used by every cell below.

In [9]:
# ---------------------------------------------------------------------------
# Freeze the exploration dataset into a structured surrogate-learning table.
#
# Each row = one full pipeline configuration that GridSearchCV evaluated.
# Columns:
#   * Categorical features  : imputer / normalizer / tamer__detector /
#                             tamer__remediation / selector / balancer /
#                             classifier-family.
#   * Continuous features   : every classifier__* and tamer__* numeric.
#   * Targets (multi-output): mean & std of PR-AUC and ROC-AUC across the 5
#                             stratified folds, plus stability proxies
#                             (range across folds, coefficient of variation,
#                             and an uncertainty-aware composite that
#                             penalises fold spread — the primary objective
#                             used downstream because medical imbalanced data
#                             rewards threshold robustness over peak PR-AUC).
#   * `_params_raw`         : the original GridSearchCV params dict, kept
#                             alongside the encoded row so the Bayesian-
#                             optimization and final-stability cells can
#                             reconstruct the actual pipeline.
#
# Calibration & threshold-sensitive target columns are created as
# placeholders. The exploration sweep only scored PR-AUC + ROC-AUC, so a
# proper Brier / recall-at-spec90 column would require re-running the
# candidates with extra scorers — left to a future sweep.
# ---------------------------------------------------------------------------
import re

def _short_repr(v):
    """Reproduce build_param_grid._short so categorical labels stay consistent
    with the diversity audit (cell 8)."""
    if v is None:
        return 'None'
    if isinstance(v, (int, float, str, bool)):
        return str(v)
    name = type(v).__name__
    for attr in ('k', 'k_neighbors', 'n_neighbors'):
        if hasattr(v, attr):
            return f'{name}({attr}={getattr(v, attr)})'
    return name


def _infer_family(params):
    """Map a candidate's classifier instance back to its family tag."""
    clf = params.get('classifier')
    if clf is None:
        return 'unknown'
    name = type(clf).__name__
    if 'XGBoost' in name:            return 'xgb'
    if 'LightGBM' in name:           return 'lgbm'
    if 'RandomForest' in name:       return 'rf'
    if 'CatBoost' in name:           return 'catboost'
    if 'LogisticRegression' in name: return 'lr'
    if 'Calibrated' in name or 'Stacking' in name: return 'stacker'
    return name.lower()


CAT_AXES = [
    'imputer', 'normalizer', 'tamer__detector', 'tamer__remediation',
    'selector', 'balancer', 'classifier__max_features', 'classifier__penalty',
]

CONT_AXES = [
    'tamer__contamination', 'tamer__mad_threshold',
    'classifier__learning_rate', 'classifier__max_depth',
    'classifier__n_estimators', 'classifier__subsample',
    'classifier__colsample_bytree', 'classifier__min_child_weight',
    'classifier__min_child_samples', 'classifier__reg_alpha',
    'classifier__reg_lambda', 'classifier__scale_pos_weight',
    'classifier__num_leaves', 'classifier__min_samples_leaf',
    'classifier__iterations', 'classifier__depth', 'classifier__l2_leaf_reg',
    'classifier__C',
    'classifier__estimator__xgb__learning_rate',
    'classifier__estimator__xgb__max_depth',
    'classifier__estimator__xgb__scale_pos_weight',
]


def build_surrogate_table(search_obj):
    """Return (meta_df, cat_cols, cont_cols, target_cols)."""
    res = pd.DataFrame(search_obj.cv_results_)
    split_cols = sorted([c for c in res.columns
                         if re.match(r'^split\d+_test_pr_auc$', c)])

    rows = []
    for i, p in enumerate(res['params']):
        row = {}
        for axis in CAT_AXES:
            row[axis] = _short_repr(p.get(axis)) if axis in p else 'absent'
        row['_family'] = _infer_family(p)
        for axis in CONT_AXES:
            v = p.get(axis, np.nan)
            row[axis] = float(v) if isinstance(v, (int, float)) and v is not None else np.nan
        row['mean_pr_auc']  = res.loc[i, 'mean_test_pr_auc']
        row['std_pr_auc']   = res.loc[i, 'std_test_pr_auc']
        row['mean_roc_auc'] = res.loc[i, 'mean_test_roc_auc']
        row['std_roc_auc']  = res.loc[i, 'std_test_roc_auc']
        if split_cols:
            fold_vals = pd.to_numeric(res.loc[i, split_cols], errors='coerce').dropna()
            row['range_pr_auc'] = float(fold_vals.max() - fold_vals.min()) if len(fold_vals) else np.nan
        else:
            row['range_pr_auc'] = np.nan
        # Keep the original params dict so BO / final-eval can reinstantiate
        # the pipeline. Stored as Python object; dropped before CSV export.
        row['_params_raw'] = p
        rows.append(row)

    meta = pd.DataFrame(rows)
    meta['cv_pr_auc'] = meta['std_pr_auc'] / np.clip(meta['mean_pr_auc'], 1e-6, None)
    meta['uncertainty_aware_score'] = meta['mean_pr_auc'] - 0.5 * meta['std_pr_auc']
    meta['brier_mean'] = np.nan
    meta['recall_at_spec90_mean'] = np.nan

    n_before = len(meta)
    meta = meta.dropna(subset=['mean_pr_auc']).reset_index(drop=True)
    print(f'Meta-table  : {len(meta)} usable configs '
          f'(dropped {n_before - len(meta)} NaN-scored).')

    cat_cols = CAT_AXES + ['_family']
    cont_cols = CONT_AXES
    target_cols = ['mean_pr_auc', 'std_pr_auc', 'mean_roc_auc', 'std_roc_auc',
                   'range_pr_auc', 'cv_pr_auc', 'uncertainty_aware_score',
                   'brier_mean', 'recall_at_spec90_mean']
    return meta, cat_cols, cont_cols, target_cols


if search is None or not hasattr(search, 'cv_results_'):
    print('[skipped] No cv_results_ available — run the exploration sweep first.')
    meta_df = None
    cat_cols = cont_cols = target_cols = None
else:
    meta_df, cat_cols, cont_cols, target_cols = build_surrogate_table(search)
    # CSV export drops object columns so the file stays portable.
    meta_df.drop(columns=['_params_raw']).to_csv('surrogate_meta_table.csv', index=False)
    print('Saved surrogate_meta_table.csv')
    print(f'  feature dims   : {len(cat_cols)} categorical + {len(cont_cols)} continuous')
    print(f'  target columns : {target_cols}')
    print()
    print('--- Family balance in the surrogate dataset ---')
    print(meta_df['_family'].value_counts().to_string())
    print()
    print('--- Primary-target spread ---')
    print(meta_df[['mean_pr_auc', 'std_pr_auc', 'cv_pr_auc',
                   'uncertainty_aware_score']].describe().round(4))
meta_df.head() if meta_df is not None else None


[skipped] No cv_results_ available — run the exploration sweep first.


## Surrogate training & analysis
Fit LightGBM / XGBoost / RandomForest / GaussianProcess / ExtraTrees on the meta-dataset and pick the surrogate with the strongest **ranking ρ** (the metric that actually matters for downstream BO). Report CV R², ρ, MAE, RMSE, predicted-vs-observed calibration, and flag low-sensitivity dimensions for freezing during exploitation.

In [10]:
# ---------------------------------------------------------------------------
# Train surrogate models on the frozen meta-dataset and audit each one.
#
# Five candidate surrogates per the project policy: LightGBM, XGBoost,
# RandomForest, GaussianProcess, ExtraTrees. We report on the meta-dataset:
#   * CV R²                  — variance explained
#   * Spearman ρ             — ranking consistency (the metric that matters
#                              for downstream BO; a surrogate that picks the
#                              right order is more useful than one that
#                              regresses the absolute score)
#   * MAE / RMSE             — error magnitude on the response
#   * predicted-vs-observed  — visual calibration of surrogate predictions
#                              against the true CV scores
#
# The surrogate with the highest ρ is kept as the working model for the
# response-surface and BO cells.
# ---------------------------------------------------------------------------
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline as SkPipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import (
    Matern, ConstantKernel as _C, WhiteKernel,
)
from sklearn.model_selection import KFold, cross_val_score
from sklearn.inspection import permutation_importance
from sklearn.metrics import r2_score
from scipy.stats import spearmanr
from lightgbm import LGBMRegressor
from xgboost import XGBRegressor
import matplotlib.pyplot as plt

PRIMARY_TARGET = 'uncertainty_aware_score'


def _make_preproc(cat_cols, cont_cols):
    return ColumnTransformer(
        transformers=[
            ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_cols),
            ('cont', SkPipeline([('imp', SimpleImputer(strategy='median'))]), cont_cols),
        ],
        remainder='drop',
    )


def _surrogate_candidates(random_state=42):
    kernel = _C(1.0) * Matern(length_scale=1.0, nu=2.5) + WhiteKernel(1e-3, (1e-6, 1.0))
    return {
        'LightGBM':     LGBMRegressor(n_estimators=400, learning_rate=0.05,
                                       num_leaves=31, min_child_samples=5,
                                       random_state=random_state, verbosity=-1),
        'XGBoost':      XGBRegressor(n_estimators=400, learning_rate=0.05,
                                      max_depth=6, subsample=0.9, colsample_bytree=0.9,
                                      random_state=random_state, verbosity=0),
        'RandomForest': RandomForestRegressor(n_estimators=500, min_samples_leaf=2,
                                               n_jobs=-1, random_state=random_state),
        'ExtraTrees':   ExtraTreesRegressor(n_estimators=500, min_samples_leaf=2,
                                             n_jobs=-1, random_state=random_state),
        'GPR':          GaussianProcessRegressor(kernel=kernel, normalize_y=True,
                                                  alpha=1e-6, random_state=random_state),
    }


def _score_surrogate(name, model, X, y):
    cv = KFold(n_splits=5, shuffle=True, random_state=42)
    r2_scores, rho_scores, maes, rmses = [], [], [], []
    for tr, te in cv.split(X):
        m = clone(model).fit(X[tr], y[tr])
        yhat = m.predict(X[te])
        r2_scores.append(r2_score(y[te], yhat))
        rho, _ = spearmanr(yhat, y[te])
        rho_scores.append(rho if np.isfinite(rho) else 0.0)
        maes.append(float(np.mean(np.abs(yhat - y[te]))))
        rmses.append(float(np.sqrt(np.mean((yhat - y[te]) ** 2))))
    return {
        'name': name,
        'r2_mean':   float(np.mean(r2_scores)),  'r2_std':   float(np.std(r2_scores)),
        'rho_mean':  float(np.mean(rho_scores)), 'rho_std':  float(np.std(rho_scores)),
        'mae_mean':  float(np.mean(maes)),       'rmse_mean': float(np.mean(rmses)),
    }


surrogate_results = None
surrogate_best = None
surrogate_best_name = None
surrogate_X = None
surrogate_y = None
preproc = None
importance_df = None
noise_dims = []

if meta_df is None or len(meta_df) < 20:
    print('[skipped] meta-table missing or too small (<20 usable rows).')
else:
    preproc = _make_preproc(cat_cols, cont_cols).fit(meta_df)
    surrogate_X = preproc.transform(meta_df).astype(np.float64)
    surrogate_y = meta_df[PRIMARY_TARGET].values

    rows = []
    print(f'--- Surrogate quality (5-fold CV; target = {PRIMARY_TARGET}) ---')
    print(f'{"model":12s}  {"R²":>14s}  {"ρ":>14s}  {"MAE":>8s}  {"RMSE":>8s}')
    for name, mdl in _surrogate_candidates().items():
        try:
            s = _score_surrogate(name, mdl, surrogate_X, surrogate_y)
            print(f'{s["name"]:12s}  {s["r2_mean"]:>+6.3f}±{s["r2_std"]:.2f}    '
                  f'{s["rho_mean"]:>+6.3f}±{s["rho_std"]:.2f}    '
                  f'{s["mae_mean"]:>6.4f}    {s["rmse_mean"]:>6.4f}')
            rows.append(s)
        except Exception as exc:
            print(f'{name:12s}  FAILED ({type(exc).__name__}: {exc})')

    surrogate_results = (pd.DataFrame(rows)
                           .sort_values('rho_mean', ascending=False)
                           .reset_index(drop=True))
    if not surrogate_results.empty:
        surrogate_best_name = surrogate_results.iloc[0]['name']
        surrogate_best = clone(_surrogate_candidates()[surrogate_best_name]).fit(
            surrogate_X, surrogate_y)
        print(f'\nSelected surrogate (by ranking ρ): {surrogate_best_name}')

# ---- predicted-vs-observed calibration plot -------------------------------
if surrogate_best is not None:
    yhat = surrogate_best.predict(surrogate_X)
    fig, ax = plt.subplots(figsize=(5, 5))
    ax.scatter(surrogate_y, yhat, alpha=0.5, s=18)
    lo, hi = float(surrogate_y.min()), float(surrogate_y.max())
    ax.plot([lo, hi], [lo, hi], color='red', lw=1, ls='--', label='y = ŷ')
    ax.set_xlabel(f'observed  ({PRIMARY_TARGET})')
    ax.set_ylabel('surrogate predicted')
    ax.set_title(f'Surrogate calibration — {surrogate_best_name}')
    ax.legend()
    plt.tight_layout(); plt.show()

# ---- Permutation importance + noise-dimension flagging --------------------
if surrogate_best is not None and surrogate_X is not None:
    feat_names = list(preproc.get_feature_names_out())
    pi = permutation_importance(surrogate_best, surrogate_X, surrogate_y,
                                n_repeats=20, random_state=42, n_jobs=-1)
    importance_df = (pd.DataFrame({'feature': feat_names,
                                   'importance': pi.importances_mean,
                                   'importance_std': pi.importances_std})
                       .sort_values('importance', ascending=False)
                       .reset_index(drop=True))
    print('\n--- Permutation importance (top 15) ---')
    print(importance_df.head(15).to_string(index=False))

    # Dimensions whose permutation drop is within their own noise band AND
    # below an absolute floor — candidates to freeze during exploitation.
    noise_mask = (
        (importance_df['importance'] <= 2 * importance_df['importance_std'])
        & (importance_df['importance'] < 0.001)
    )
    noise_dims = importance_df.loc[noise_mask, 'feature'].tolist()
    print(f'\nLow-sensitivity dims flagged for freezing during BO: {len(noise_dims)}')
    for d in noise_dims[:10]:
        print(f'  · {d}')
    importance_df.to_csv('surrogate_importance.csv', index=False)

    # Optional SHAP (gracefully skipped if not installed).
    try:
        import shap  # type: ignore
        sample = surrogate_X[:min(200, len(surrogate_X))]
        try:
            explainer = shap.TreeExplainer(surrogate_best)
            sv = explainer.shap_values(sample)
        except Exception:
            explainer = shap.Explainer(surrogate_best.predict, sample)
            sv = explainer(sample).values
        shap_summary = (pd.Series(np.abs(sv).mean(0), index=feat_names)
                          .sort_values(ascending=False))
        print('\n--- |SHAP| mean (top 10) ---')
        print(shap_summary.head(10).to_string())
    except Exception as exc:
        print(f'\n[shap] unavailable — using permutation importance only '
              f'({type(exc).__name__}).')


[skipped] meta-table missing or too small (<20 usable rows).


## Response surface
Visualize the surrogate-predicted PR-AUC surface over the top continuous dimension pairs (ordered by permutation importance) with the actual evaluated configs overlaid, plus categorical projections — exposes smooth regions, cliffs, plateaus, and interaction shapes.

In [11]:
# ---------------------------------------------------------------------------
# Response-surface visualizations over the strongest dimensions.
#
# For each pair of top continuous knobs (ordered by permutation importance):
#   * surrogate-predicted PR-AUC contour on a 30×30 grid, with other features
#     held at their medians / modes — this exposes smooth regions, cliffs,
#     plateaus, and interaction shapes;
#   * overlay of actual evaluated configurations coloured by true mean PR-AUC,
#     showing how the explored cloud sits on the predicted surface.
#
# For categorical knobs (selector, taming remediation, model family,
# balancer) we project PR-AUC distributions as boxplots — a flat boxplot
# means the category is effectively noise on this dataset.
# ---------------------------------------------------------------------------
import matplotlib.pyplot as plt
import seaborn as sns
from itertools import combinations

KEY_CONT = [
    'classifier__learning_rate', 'classifier__scale_pos_weight',
    'classifier__max_depth', 'classifier__reg_alpha', 'classifier__reg_lambda',
    'classifier__subsample', 'classifier__colsample_bytree',
    'classifier__n_estimators',
]
KEY_CAT = ['selector', 'tamer__remediation', '_family', 'balancer']


def _importance_for(col_substr, imp_df):
    """Sum permutation importances of all OHE features matching the substring."""
    if imp_df is None:
        return 0.0
    return float(imp_df.loc[imp_df['feature'].str.contains(col_substr, regex=False),
                            'importance'].sum())


def _template_row(meta_df, cat_cols, cont_cols):
    row = {}
    for c in cont_cols:
        row[c] = float(meta_df[c].median())
    for c in cat_cols:
        row[c] = meta_df[c].mode().iloc[0]
    return row


if surrogate_best is None or meta_df is None:
    print('[skipped] no surrogate / meta-table.')
else:
    available_cont = [c for c in KEY_CONT
                      if c in meta_df.columns
                      and meta_df[c].notna().sum() >= 20
                      and float(meta_df[c].std() or 0) > 0]
    available_cont.sort(key=lambda c: _importance_for(c, importance_df), reverse=True)
    top_pairs = list(combinations(available_cont[:4], 2))[:3]

    if top_pairs:
        fig, axes = plt.subplots(1, len(top_pairs),
                                 figsize=(5 * len(top_pairs), 4.2), squeeze=False)
        template = _template_row(meta_df, cat_cols, cont_cols)

        for ax, (a, b) in zip(axes[0], top_pairs):
            grid_a = np.linspace(meta_df[a].min(), meta_df[a].max(), 30)
            grid_b = np.linspace(meta_df[b].min(), meta_df[b].max(), 30)
            Ga, Gb = np.meshgrid(grid_a, grid_b)

            rows = []
            for av, bv in zip(Ga.ravel(), Gb.ravel()):
                t = dict(template); t[a] = av; t[b] = bv
                rows.append(t)
            grid_df = pd.DataFrame(rows)[cat_cols + cont_cols]
            Xq = preproc.transform(grid_df).astype(np.float64)
            Z = surrogate_best.predict(Xq).reshape(Ga.shape)

            im = ax.contourf(Ga, Gb, Z, levels=20, cmap='viridis')
            ax.scatter(meta_df[a], meta_df[b],
                       c=meta_df['mean_pr_auc'], cmap='magma',
                       edgecolor='white', linewidth=0.4, s=20)
            ax.set_xlabel(a.split('__')[-1])
            ax.set_ylabel(b.split('__')[-1])
            ax.set_title(f'{a.split("__")[-1]} × {b.split("__")[-1]}', fontsize=10)
            fig.colorbar(im, ax=ax, shrink=0.8)
        fig.suptitle(f'Surrogate-predicted {PRIMARY_TARGET} surface  '
                     f'(dots = evaluated configs, colour = true mean_pr_auc)',
                     y=1.04, fontsize=11)
        plt.tight_layout(); plt.show()
    else:
        print('[note] not enough continuous dispersion for response-surface plot.')

    # Categorical projections
    cat_present = [c for c in KEY_CAT
                   if c in meta_df.columns and meta_df[c].nunique() > 1]
    if cat_present:
        fig, axes = plt.subplots(1, len(cat_present),
                                 figsize=(4.2 * len(cat_present), 4), squeeze=False)
        for ax, c in zip(axes[0], cat_present):
            sns.boxplot(data=meta_df, x=c, y='mean_pr_auc', ax=ax,
                        order=sorted(meta_df[c].dropna().unique().tolist()))
            ax.set_title(c, fontsize=10)
            for lbl in ax.get_xticklabels():
                lbl.set_rotation(30); lbl.set_horizontalalignment('right')
        plt.tight_layout(); plt.show()


[skipped] no surrogate / meta-table.


## Bayesian optimization
Staged-batch BO over the surrogate: UCB/EI acquisition with ExtraTrees per-tree variance, **minimum diversity quotas** per batch across family/selector/balancer, **uncertainty-aware objective** (mean PR-AUC − 0.5·std PR-AUC), surrogate retrained between batches, and a trust-region local-refinement pass around the best config. Flag-gated by `RUN_BO` because it re-evaluates pipelines via 5-fold CV.

In [12]:
# ---------------------------------------------------------------------------
# Bayesian optimization over the surrogate.
#
# Policy (see memory):
#   * Run BO only after exploration produced sufficient entropy and the
#     surrogate has acceptable quality (this cell trusts the previous cells).
#   * Staged batches with surrogate retraining between batches.
#   * Acquisition: EI or UCB over the surrogate (uncertainty estimated by
#     ExtraTrees per-tree variance — gives us μ + σ from a single fit).
#   * Preserve diversity: minimum quotas across model family, selector, and
#     balancer in every batch — never collapse onto one regime.
#   * Uncertainty-aware objective: mean_pr_auc − λ·std_pr_auc, NOT raw PR-AUC.
#     Threshold robustness matters more than peak score on imbalanced
#     medical data.
#   * Local refinement: a trust-region step around the best config after the
#     last batch, with continuous ranges shrunk to ±TR_SHRINK around best and
#     categoricals pinned.
#
# Heavy step (re-fits pipelines via 5-fold CV) — gated by RUN_BO.
# ---------------------------------------------------------------------------
RUN_BO = False   # opt-in; flip to True to launch the loop.

BO_BATCHES               = 3
BO_BATCH_SIZE            = 8
BO_POOL_PER_BATCH        = 400
BO_ACQUISITION           = 'ucb'      # 'ucb' or 'ei'
BO_KAPPA                 = 1.5        # UCB exploration weight
BO_XI                    = 0.001      # EI exploration buffer
BO_FAMILY_MIN_FRACTION   = 0.10       # min slots/batch per active family
BO_SELECTOR_MIN_FRACTION = 0.05
BO_BALANCER_MIN_FRACTION = 0.05
TR_N                     = 12         # trust-region samples after batches
TR_SHRINK                = 0.4        # ±40% of original range, log-aware
RANDOM_STATE             = 42

from sklearn.base import clone
from sklearn.model_selection import cross_validate, StratifiedKFold
from scipy.stats import norm


def _acquisition(mu, sigma, f_best, kind, kappa=BO_KAPPA, xi=BO_XI):
    if kind == 'ucb':
        return mu + kappa * sigma
    sigma = np.clip(sigma, 1e-9, None)
    imp = mu - f_best - xi
    z = imp / sigma
    return imp * norm.cdf(z) + sigma * norm.pdf(z)


def _bo_surrogate_fit(X, y, random_state=RANDOM_STATE):
    from sklearn.ensemble import ExtraTreesRegressor
    et = ExtraTreesRegressor(n_estimators=500, min_samples_leaf=2,
                              n_jobs=-1, random_state=random_state)
    et.fit(X, y)
    return et


def _bo_surrogate_predict(et, X):
    preds = np.array([t.predict(X) for t in et.estimators_])
    return preds.mean(0), preds.std(0)


def _encode_param_dicts(param_dicts, family_list, preproc_, cat_cols_, cont_cols_):
    """Convert build_param_grid-style dicts (length-1-list values) into the
    same encoded feature matrix the surrogate sees."""
    rows = []
    for p, fam in zip(param_dicts, family_list):
        row = {}
        for axis in cat_cols_:
            if axis == '_family':
                row[axis] = fam
                continue
            v = p.get(axis, [None])
            v = v[0] if isinstance(v, list) else v
            row[axis] = _short_repr(v) if v is not None else 'absent'
        for axis in cont_cols_:
            v = p.get(axis, [np.nan])
            v = v[0] if isinstance(v, list) else v
            row[axis] = float(v) if isinstance(v, (int, float)) and v is not None else np.nan
        rows.append(row)
    df = pd.DataFrame(rows)[cat_cols_ + cont_cols_]
    return preproc_.transform(df).astype(np.float64), df


def _select_with_quotas(acq_scores, families, selectors, balancers, batch_size,
                        fam_min_frac, sel_min_frac, bal_min_frac):
    """Greedy top-by-acquisition pick, then repair to satisfy minimum-quota
    constraints by swapping the lowest-acquisition chosen with the highest-
    acquisition under-represented candidate."""
    order = np.argsort(-acq_scores)
    chosen = list(order[:batch_size])

    def _enforce(attr_vals, min_frac, label):
        nonlocal chosen
        quota = max(1, int(round(min_frac * batch_size)))
        from collections import Counter
        counts = Counter(attr_vals[i] for i in chosen)
        under = [v for v in set(attr_vals) if counts.get(v, 0) < quota]
        for u in under:
            pool = [i for i in order
                    if attr_vals[i] == u and i not in chosen]
            if not pool:
                continue
            # Drop the lowest-acquisition chosen that's NOT under-represented.
            drop_candidates = sorted(
                [i for i in chosen if counts[attr_vals[i]] > quota],
                key=lambda i: acq_scores[i])
            if not drop_candidates:
                continue
            needed = quota - counts.get(u, 0)
            for j_take, j_drop in zip(pool[:needed], drop_candidates[:needed]):
                chosen.remove(j_drop)
                chosen.append(j_take)
                counts[attr_vals[j_drop]] -= 1
                counts[u] = counts.get(u, 0) + 1
    _enforce(np.asarray(families),  fam_min_frac, 'family')
    _enforce(np.asarray(selectors), sel_min_frac, 'selector')
    _enforce(np.asarray(balancers), bal_min_frac, 'balancer')
    return chosen


def _evaluate_candidate(params_dict, pipeline_, x_train_, y_train_, cv_):
    """Run 5-fold CV for one candidate; return summary metrics or NaN row."""
    cfg = {k: (v[0] if isinstance(v, list) else v) for k, v in params_dict.items()}
    pipe = clone(pipeline_)
    try:
        pipe.set_params(**cfg)
        cvres = cross_validate(
            pipe, x_train_, y_train_,
            scoring={'pr_auc': 'average_precision', 'roc_auc': 'roc_auc'},
            cv=cv_, n_jobs=-1, error_score=np.nan, return_train_score=False,
        )
        pr = np.asarray(cvres['test_pr_auc'])
        rc = np.asarray(cvres['test_roc_auc'])
        return {
            'mean_pr_auc':  float(np.nanmean(pr)),
            'std_pr_auc':   float(np.nanstd(pr)),
            'mean_roc_auc': float(np.nanmean(rc)),
            'std_roc_auc':  float(np.nanstd(rc)),
            'range_pr_auc': float(np.nanmax(pr) - np.nanmin(pr)),
        }
    except Exception as exc:
        print(f'    [eval-fail] {type(exc).__name__}: {exc}')
        return None


def _build_trust_region_pool(best_row, n, shrink, spw_base_):
    """Sample around the best config: categoricals pinned, continuous ranges
    contracted to ±shrink·range_original around the best value."""
    from build_param_grid import (
        build_param_grid as _bpg, PREPROC_DIMS,
        _xgb_dims, _lgbm_dims, _rf_dims, _catboost_dims, _lr_dims, _stacker_dims,
        _sample_family, _row_to_dict, _lhs,
    )
    # Reuse the family-specific dim list, but clip continuous bounds.
    fam = best_row['_family']
    spw_cap = float(min(10.0, 1.5 * spw_base_))
    fam_dims_fn = {
        'xgb':      lambda: _xgb_dims(spw_cap),
        'lgbm':     lambda: _lgbm_dims(spw_cap),
        'rf':       lambda: _rf_dims(),
        'catboost': lambda: _catboost_dims(),
        'lr':       lambda: _lr_dims(),
        'stacker':  lambda: _stacker_dims(spw_base_),
    }.get(fam, lambda: _xgb_dims(spw_cap))
    dims = PREPROC_DIMS + fam_dims_fn()

    # Clip continuous bounds around the best value when the row has it.
    clipped = []
    for spec in dims:
        name, kind = spec[0], spec[1]
        v_best = best_row.get(name)
        if kind in ('lin', 'log') and isinstance(v_best, (int, float)) and np.isfinite(v_best):
            lo, hi = spec[2], spec[3]
            span = (hi - lo) if kind == 'lin' else (np.log(hi) - np.log(lo))
            half = 0.5 * shrink * span
            if kind == 'lin':
                new_lo, new_hi = max(lo, v_best - half), min(hi, v_best + half)
            else:
                new_lo = max(lo, float(np.exp(np.log(v_best) - half)))
                new_hi = min(hi, float(np.exp(np.log(v_best) + half)))
            if new_hi > new_lo:
                clipped.append((name, kind, new_lo, new_hi))
                continue
        clipped.append(spec)

    samples = _sample_family(fam, clipped, n, seed=RANDOM_STATE + 999)
    return samples, [fam] * n


if not RUN_BO:
    print('[skipped] RUN_BO=False — flip and re-run to launch Bayesian optimization.')
    bo_results = None
elif surrogate_best is None or preproc is None:
    print('[skipped] surrogate / preprocessor not available.')
    bo_results = None
else:
    from build_param_grid import build_param_grid as _bpg

    cv5 = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    cur_meta = meta_df.copy()
    cur_X = preproc.transform(cur_meta).astype(np.float64)
    cur_y = cur_meta[PRIMARY_TARGET].values
    f_best = float(cur_y.max())
    print(f'BO start: baseline n={len(cur_meta)}, f_best={f_best:.4f}')

    bo_log = []

    for b in range(BO_BATCHES):
        pool_params, pool_family = _bpg(
            spw_base, total_n=BO_POOL_PER_BATCH, seed=RANDOM_STATE + 100 + b,
        )
        bo_surr = _bo_surrogate_fit(cur_X, cur_y)
        pool_X, pool_df = _encode_param_dicts(
            pool_params, pool_family, preproc, cat_cols, cont_cols,
        )
        mu, sigma = _bo_surrogate_predict(bo_surr, pool_X)
        acq = _acquisition(mu, sigma, f_best, BO_ACQUISITION)
        sel_idx = _select_with_quotas(
            acq, pool_family,
            pool_df['selector'].astype(str).values,
            pool_df['balancer'].astype(str).values,
            BO_BATCH_SIZE,
            BO_FAMILY_MIN_FRACTION, BO_SELECTOR_MIN_FRACTION, BO_BALANCER_MIN_FRACTION,
        )

        for j in sel_idx:
            metrics = _evaluate_candidate(
                pool_params[j], pipeline, x_train, y_train, cv5,
            )
            if metrics is None:
                continue
            row = pool_df.iloc[j].to_dict()
            row.update(metrics)
            row['cv_pr_auc'] = row['std_pr_auc'] / max(row['mean_pr_auc'], 1e-6)
            row['uncertainty_aware_score'] = row['mean_pr_auc'] - 0.5 * row['std_pr_auc']
            row['brier_mean'] = np.nan
            row['recall_at_spec90_mean'] = np.nan
            row['_family'] = pool_family[j]
            row['_batch'] = b
            row['_params_raw'] = pool_params[j]
            bo_log.append(row)

        if bo_log:
            cur_meta = pd.concat(
                [meta_df.assign(_batch=-1),
                 pd.DataFrame(bo_log)],
                ignore_index=True,
            )
            cur_X = preproc.transform(cur_meta).astype(np.float64)
            cur_y = cur_meta[PRIMARY_TARGET].values
            f_best = float(np.nanmax(cur_y))
        print(f'  batch {b+1}/{BO_BATCHES}: '
              f'n={len(cur_meta)}, f_best={f_best:.4f}')

    # ---- Trust-region local refinement around the best config ------------
    best_row = cur_meta.loc[cur_meta[PRIMARY_TARGET].idxmax()].to_dict()
    print(f'\nTrust-region refinement around best (family={best_row["_family"]}, '
          f'score={best_row[PRIMARY_TARGET]:.4f})')
    tr_params, tr_family = _build_trust_region_pool(
        best_row, TR_N, TR_SHRINK, spw_base,
    )
    for j, p in enumerate(tr_params):
        metrics = _evaluate_candidate(p, pipeline, x_train, y_train, cv5)
        if metrics is None:
            continue
        # Encode for downstream use.
        tr_X, tr_df = _encode_param_dicts(
            [p], [tr_family[j]], preproc, cat_cols, cont_cols,
        )
        row = tr_df.iloc[0].to_dict()
        row.update(metrics)
        row['cv_pr_auc'] = row['std_pr_auc'] / max(row['mean_pr_auc'], 1e-6)
        row['uncertainty_aware_score'] = row['mean_pr_auc'] - 0.5 * row['std_pr_auc']
        row['brier_mean'] = np.nan
        row['recall_at_spec90_mean'] = np.nan
        row['_family'] = tr_family[j]
        row['_batch'] = 'TR'
        row['_params_raw'] = p
        bo_log.append(row)

    bo_results = pd.DataFrame(bo_log)
    if len(bo_results):
        combined = pd.concat(
            [meta_df.assign(_batch=-1),
             bo_results],
            ignore_index=True,
        )
        combined.drop(columns=['_params_raw']).to_csv(
            'surrogate_meta_table_after_bo.csv', index=False,
        )
        final_best = combined.loc[combined[PRIMARY_TARGET].idxmax()]
        print(f'\nBO complete. Cumulative best uncertainty_aware = '
              f'{final_best[PRIMARY_TARGET]:.4f}, family={final_best["_family"]}.')
        bo_results = combined  # downstream cells consume the full table


[skipped] RUN_BO=False — flip and re-run to launch Bayesian optimization.


## Final stability evaluation
Top-K candidates are re-evaluated with **RepeatedStratifiedKFold** (5 folds × 3 repeats) — PR-AUC mean/std, Brier (calibration), recall at Spec ≥ 0.90, and best-F1 threshold spread across folds. A pipeline is only fit to freeze for deployment if it passes this gate. Flag-gated by `RUN_FINAL_EVAL`.

In [13]:
# ---------------------------------------------------------------------------
# Final stability gate.
#
# Pick the top-K candidates from whichever table is freshest (BO results if
# available, else the exploration meta-table) and re-evaluate them with
# RepeatedStratifiedKFold. For each candidate we report:
#   * PR-AUC mean + std across all repeats
#   * Brier score (calibration)
#   * Recall at Spec ≥ 0.90 (clinically meaningful operating point)
#   * Best-F1 threshold spread across folds (threshold stability)
#
# A pipeline is only fit to be frozen if it passes this gate. Aggregate
# PR-AUC from the 5-fold exploration sweep is NOT sufficient for deployment.
#
# Heavy step — gated by RUN_FINAL_EVAL.
# ---------------------------------------------------------------------------
RUN_FINAL_EVAL = False
TOP_K          = 5
FINAL_FOLDS    = 5
N_REPEATS      = 3
SPEC_TARGET    = 0.90

from sklearn.base import clone
from sklearn.model_selection import RepeatedStratifiedKFold
from sklearn.metrics import (
    average_precision_score, roc_auc_score, brier_score_loss,
    precision_recall_curve, roc_curve,
)

if not RUN_FINAL_EVAL:
    print('[skipped] RUN_FINAL_EVAL=False — flip to launch the stability gate.')
    final_stability_df = None
    final_winner_params = None
else:
    source = bo_results if (
        'bo_results' in dir() and bo_results is not None and len(bo_results) > 0
    ) else meta_df
    if source is None or len(source) == 0:
        print('[skipped] No candidate source available.')
        final_stability_df = None
        final_winner_params = None
    else:
        top = (source.sort_values('uncertainty_aware_score', ascending=False)
                     .head(TOP_K)
                     .reset_index(drop=True))
        rcv = RepeatedStratifiedKFold(
            n_splits=FINAL_FOLDS, n_repeats=N_REPEATS, random_state=42,
        )
        # Preserve x_train's container type — the custom transformers in
        # tame_outlier / normalization / feature_selection record column
        # names at fit time, so we must slice with .iloc to keep the
        # DataFrame structure that the original GridSearchCV fit used.
        is_df = hasattr(x_train, 'iloc')
        yt_arr = np.asarray(y_train)
        # rcv.split only needs row counts + y for stratification.
        split_X = x_train.values if is_df else np.asarray(x_train)

        rows = []
        for rank in range(len(top)):
            raw = top.loc[rank, '_params_raw']
            if raw is None or (isinstance(raw, float) and not np.isfinite(raw)):
                print(f'  #{rank}: no _params_raw — skipped.')
                continue
            cfg = {k: (v[0] if isinstance(v, list) else v) for k, v in raw.items()}
            try:
                base_pipe = clone(pipeline).set_params(**cfg)
            except Exception as exc:
                print(f'  #{rank}: set_params failed ({exc.__class__.__name__}: {exc})')
                continue

            pr_aucs, roc_aucs, briers = [], [], []
            recalls_at_spec, f1_thresholds = [], []
            for tr_idx, te_idx in rcv.split(split_X, yt_arr):
                if is_df:
                    X_tr, X_te = x_train.iloc[tr_idx], x_train.iloc[te_idx]
                else:
                    X_tr, X_te = split_X[tr_idx], split_X[te_idx]
                y_tr, y_te = yt_arr[tr_idx], yt_arr[te_idx]
                try:
                    m = clone(base_pipe).fit(X_tr, y_tr)
                    proba = m.predict_proba(X_te)[:, 1]
                except Exception as exc:
                    print(f'    fold-fail rank={rank}: '
                          f'{exc.__class__.__name__}: {exc}')
                    continue
                pr_aucs.append(average_precision_score(y_te, proba))
                roc_aucs.append(roc_auc_score(y_te, proba))
                briers.append(brier_score_loss(y_te, proba))
                # Best-F1 threshold this fold
                precs, recs, ths = precision_recall_curve(y_te, proba)
                f1 = 2 * precs[:-1] * recs[:-1] / np.clip(
                    precs[:-1] + recs[:-1], 1e-12, None,
                )
                f1_thresholds.append(
                    float(ths[int(np.nanargmax(f1))]) if len(f1) else np.nan
                )
                # Recall at Spec >= SPEC_TARGET
                fpr, tpr, _ = roc_curve(y_te, proba)
                mask = (1 - fpr) >= SPEC_TARGET
                recalls_at_spec.append(
                    float(np.max(tpr * mask)) if mask.any() else np.nan
                )

            if not pr_aucs:
                continue
            rows.append({
                'rank':                  rank,
                'family':                top.loc[rank, '_family'],
                'pr_auc_mean':           float(np.mean(pr_aucs)),
                'pr_auc_std':            float(np.std(pr_aucs)),
                'roc_auc_mean':          float(np.mean(roc_aucs)),
                'roc_auc_std':           float(np.std(roc_aucs)),
                'brier_mean':            float(np.mean(briers)),
                'brier_std':             float(np.std(briers)),
                f'recall@spec{int(SPEC_TARGET*100)}_mean': float(np.nanmean(recalls_at_spec)),
                f'recall@spec{int(SPEC_TARGET*100)}_std':  float(np.nanstd(recalls_at_spec)),
                'thr_f1_mean':           float(np.nanmean(f1_thresholds)),
                'thr_f1_std':            float(np.nanstd(f1_thresholds)),
                'n_folds':               len(pr_aucs),
            })

        final_stability_df = (pd.DataFrame(rows)
                                .sort_values('pr_auc_mean', ascending=False)
                                .reset_index(drop=True))
        print('\n--- Final stability evaluation '
              f'({FINAL_FOLDS}×{N_REPEATS} RepeatedStratifiedKFold) ---')
        if len(final_stability_df):
            print(final_stability_df.to_string(index=False))
            final_stability_df.to_csv('final_stability.csv', index=False)

            # Stability-aware winner — penalise PR-AUC spread and Brier.
            final_stability_df['stability_score'] = (
                final_stability_df['pr_auc_mean']
                - 0.5 * final_stability_df['pr_auc_std']
                - 0.5 * final_stability_df['brier_mean']
            )
            winner_rank = int(final_stability_df
                              .sort_values('stability_score', ascending=False)
                              .iloc[0]['rank'])
            final_winner_params = top.loc[winner_rank, '_params_raw']
            print(f'\n>>> Final stability winner: rank #{winner_rank} '
                  f'(family={top.loc[winner_rank, "_family"]})')

            # ----------------------------------------------------------------
            # Freeze the stability winner alongside the GridSearch winner.
            # Distinct filenames (`*_stable.pkl`) so the original persist cell
            # remains untouched and either artifact can be deployed.
            # ----------------------------------------------------------------
            try:
                cfg_w = {k: (v[0] if isinstance(v, list) else v)
                         for k, v in final_winner_params.items()}
                stable_pipe = clone(pipeline).set_params(**cfg_w).fit(x_train, y_train)
                proba_w = stable_pipe.predict_proba(x_val)[:, 1]
                # Recompute holdout thresholds for this winner.
                precs_w, recs_w, ths_w = precision_recall_curve(y_val, proba_w)
                f1_w = 2 * precs_w[:-1] * recs_w[:-1] / np.clip(
                    precs_w[:-1] + recs_w[:-1], 1e-12, None,
                )
                thr_f1_w = float(ths_w[int(np.nanargmax(f1_w))]) if len(f1_w) else float('nan')
                fpr_w, tpr_w, rth_w = roc_curve(y_val, proba_w)
                mask_w = (1 - fpr_w) >= SPEC_TARGET
                if mask_w.any():
                    idx_w = int(np.argmax(tpr_w * mask_w))
                    thr_spec_w = float(rth_w[idx_w]); rec_spec_w = float(tpr_w[idx_w])
                else:
                    thr_spec_w = float('nan'); rec_spec_w = float('nan')

                out_dir = os.path.dirname(path) or '.'
                os.makedirs(out_dir, exist_ok=True)
                joblib.dump(stable_pipe, os.path.join(out_dir, 'pipeline_stable.pkl'))
                joblib.dump({
                    'threshold_f1':                              thr_f1_w,
                    f'threshold_at_spec{int(SPEC_TARGET*100)}':  thr_spec_w,
                    f'recall_at_spec{int(SPEC_TARGET*100)}':     rec_spec_w,
                    'val_pr_auc':                                float(average_precision_score(y_val, proba_w)),
                    'val_roc_auc':                               float(roc_auc_score(y_val, proba_w)),
                    'winner_family':                             top.loc[winner_rank, '_family'],
                    'winner_stability_score':                    float(final_stability_df['stability_score'].max()),
                }, os.path.join(out_dir, 'operating_point_stable.pkl'))
                print(f'Saved: {os.path.join(out_dir, "pipeline_stable.pkl")}')
                print(f'Saved: {os.path.join(out_dir, "operating_point_stable.pkl")}')
            except Exception as exc:
                print(f'[freeze-fail] {type(exc).__name__}: {exc}')
        else:
            print('  (no successful evaluations)')
            final_winner_params = None


[skipped] RUN_FINAL_EVAL=False — flip to launch the stability gate.


## Persist artifacts

In [14]:
if search is None or not hasattr(search, 'best_estimator_'):
    print('[skipped] No fitted estimator to persist.')
else:
    out_dir = os.path.dirname(path) or '.'
    os.makedirs(out_dir, exist_ok=True)

    joblib.dump(search.best_estimator_, os.path.join(out_dir, 'pipeline.pkl'))
    joblib.dump(all_mappings,           os.path.join(out_dir, 'all_mapping.pkl'))
    joblib.dump(y_mappings,             os.path.join(out_dir, 'y_mappings.pkl'))
    joblib.dump(list(X.columns),        os.path.join(out_dir, 'features.pkl'))

    # Persist the tuned operating point — required for inference, since the
    # default 0.5 is the wrong cut for this prevalence.
    joblib.dump(
        {
            'threshold_f1':     best_thr,
            'threshold_f2':     best_f2_thr,
            'threshold_spec90': thr_at_spec90,
            'val_roc_auc':      val_roc_auc,
            'val_pr_auc':       val_pr_auc,
            'recall_at_spec90': recall_at_spec90,
        },
        os.path.join(out_dir, 'operating_point.pkl'),
    )

    print('Saved:', os.path.join(out_dir, 'pipeline.pkl'))
    print('Saved:', os.path.join(out_dir, 'operating_point.pkl'))

[skipped] No fitted estimator to persist.


In [15]:
import joblib
import os

out_dir = "./CardioKaggle/"

pipeline = joblib.load(os.path.join(out_dir, "pipeline.pkl"))
all_mappings = joblib.load(os.path.join(out_dir, "all_mapping.pkl"))
y_mappings = joblib.load(os.path.join(out_dir, "y_mappings.pkl"))
features = joblib.load(os.path.join(out_dir, "features.pkl"))
operating_point = joblib.load(os.path.join(out_dir, "operating_point.pkl"))

print("=== PIPELINE ===")
print(pipeline)

print("\n=== ALL MAPPINGS ===")
print(all_mappings)

print("\n=== Y MAPPINGS ===")
print(y_mappings)

print("\n=== FEATURES ===")
print(features)

print("\n=== OPERATING POINT ===")
print(operating_point)

FileNotFoundError: [Errno 2] No such file or directory: './CardioKaggle/pipeline.pkl'